In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from config_train import get_config
from torch.utils.data import DataLoader
from tsp import TSPDataset
from cvrp import CVRPDataset
import torch
import numpy as np
import time
import datetime
import tqdm
import os
import logging
import sys
from utils import read_instance_data
# import importlib
# importlib.reload(sys.modules['search_control'])
# importlib.reload(sys.modules['de'])
from search_control import solve_instance_set
from VAE_8 import VAE_8
torch.set_float32_matmul_precision('high')
from torch.utils.tensorboard import SummaryWriter

def calculate_RC_loss(tour_logp):
    RC = - tour_logp.sum() / 2
    return RC


def calculate_KLD_loss(mean, log_var):
    KLD = -0.5 * torch.sum(1 + log_var - mean.pow(2) - log_var.exp()) / 2
    return KLD

In [3]:
def evaluate_network(config, model, validation_dataloader, epoch_idx):
    model.eval()
    loss_RC_values = []
    loss_KLD_values = []
    abs_Z_values = []
    for batch_id, batch in enumerate(validation_dataloader):
        instances, solutions_1, solutions_2 = batch

        with torch.no_grad():
            output, mean, log_var, Z, tour_idx, tour_logp = model(instances, solutions_1, solutions_2, config)
        loss_RC = calculate_RC_loss(tour_logp)
        loss_KLD = calculate_KLD_loss(mean, log_var)

        loss_RC_values.append(loss_RC.item())
        loss_KLD_values.append(loss_KLD.item())
        abs_Z = torch.abs(Z)  # Absolute coordinates of points in latent space (Z)
        abs_Z_values.append(abs_Z.cpu().numpy())

    abs_Z_values = np.array(abs_Z_values).flatten()

    # The bounds of the search space are defined as a percentile of the absolute latent variable coordinates
    new_bound = np.percentile(abs_Z_values, config.q_percentile).item()

    avgRC = np.mean(loss_RC_values)
    avgKL = np.mean(loss_KLD_values)
    writer.add_scalar("Loss/Reconstruction/Val", avgRC, epoch_idx)
    writer.add_scalar("Loss/KL-divergence/Val", avgKL, epoch_idx)
    writer.add_scalar("Loss/Combined/Val", avgRC + config.KLD_weight * avgKL, epoch_idx)

    return new_bound

In [4]:
VERSION = "0.4.0"
run_id = np.random.randint(10000, 99999)
now = datetime.datetime.now()

config = get_config(inJupyter=True)

if config.output_path == "":
    config.output_path = os.getcwd()
run_id = f"run_{now.day}.{now.month}_{now.hour}-{now.minute}-{now.second}_{run_id}"
config.output_path = os.path.join(config.output_path, "runs", run_id)
os.makedirs(os.path.join(config.output_path, "models"))

writer = SummaryWriter(log_dir=f"tensorboard_logdir/{run_id}")
writer.add_text('config', '\n'.join(map(lambda k_v: f"{k_v[0]}: {k_v[1]}", vars(config).items())))

logging.basicConfig(
    filename=os.path.join(config.output_path, "log_" + str(run_id) + ".txt"), filemode='w',
    level=logging.INFO, format='[%(levelname)s]%(message)s', force=True)
logging.info("Started Training Run")
logging.info("Call: {0}".format(''.join(sys.argv)))
logging.info("Version: {0}".format(VERSION))
logging.info("PARAMETERS:")
for arg in sorted(vars(config)):
    logging.info("{0}: {1}".format(arg, getattr(config, arg)))
logging.info("----------")
training_data, validation_data = read_instance_data(config) # this takes 10 fucking seconds...

In [5]:
def train_step(instances, solutions_1, solutions_2):
    optimizer.zero_grad()
    output, mean, log_var, Z, tour_idx, tour_logp = model(instances, solutions_1, solutions_2, config)
    loss_RC = calculate_RC_loss(tour_logp)
    loss_KLD = calculate_KLD_loss(mean, log_var)
    loss = loss_RC + loss_KLD * config.KLD_weight
    assert not torch.isnan(loss)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
    optimizer.step()
    return loss_RC, loss_KLD

def train_epoch(epoch_idx):
    global training_instances, training_solutions
    model.train()
    # n = training_instances.shape[0]
    # perm = torch.randperm(n)
    # training_instances = training_instances[perm, :]
    # training_solutions = training_solutions[perm, :]
    # shift1 = torch.randint(high=config.problem_size, size=(1,))
    # s1 = torch.roll(training_solutions, int(shift1), 1)
    # shift2 = torch.randint(high=config.problem_size, size=(1,))
    # s2 = torch.roll(training_solutions, int(shift2), 1)

    # for batch_id in range((training_instances.shape[0] + config.batch_size - 1) // config.batch_size):
    #     sl = slice(batch_id * config.batch_size, (batch_id + 1) * config.batch_size)
    #     instances, solutions_1, solutions_2 = training_instances[sl], s1[sl], s2[sl]
    loss_RC_values = []
    loss_KLD_values = []
    for batch_id, batch in enumerate(training_dataloader):
        instances, solutions_1, solutions_2 = batch
        loss_RC, loss_KLD = train_step(instances, solutions_1, solutions_2)
        loss_RC_values.append(loss_RC.item())
        loss_KLD_values.append(loss_KLD.item())
        lr_scheduler.step()
    avgRC = np.mean(loss_RC_values)
    avgKL = np.mean(loss_KLD_values)
    writer.add_scalar("Loss/Reconstruction/Train", avgRC, epoch_idx)
    writer.add_scalar("Loss/KL-divergence/Train", avgKL, epoch_idx)
    writer.add_scalar("Loss/Combined/Train", avgRC + config.KLD_weight * avgKL, epoch_idx)
    writer.add_scalar("LR", lr_scheduler.get_last_lr()[0], epoch_idx)

In [6]:
def do(starting_epoch = 0):
    best_avg_gap = np.inf
    for epoch_idx in tqdm.trange(1, config.nb_epochs + 1):
        epoch_idx += starting_epoch
        train_epoch(epoch_idx)
        if epoch_idx % 50 == 0:
            new_bound = evaluate_network(config, model, validation_dataloader, epoch_idx)

            config.search_space_bound = new_bound
            writer.add_scalar("Search Space Bounds", new_bound, epoch_idx)

            avg_gap, avg_runtime, _ = solve_instance_set(model, config,
                                                         validation_data[0][: config.search_validation_size]
                                                         , validation_data[1][:config.search_validation_size])

            # If the average gap is improved, save the model
            if avg_gap < best_avg_gap:
                best_avg_gap = avg_gap
                model_data = {
                    'parameters': model.state_dict(),
                    'code_version': VERSION,
                    'problem': config.problem,
                    'problem_size': config.problem_size,
                    'Z_bound': new_bound,
                    'avg_gap': avg_gap,
                    'training_epochs': epoch_idx,
                    'model': "VAE_final"
                }

                torch.save(model_data, os.path.join(config.output_path, "models",
                                                    "model_{0}.pt".format(run_id, epoch_idx)))

            writer.add_scalar("Gap/Val", avg_gap * 100, epoch_idx)
            writer.add_scalar("Search Time/Val", avg_runtime, epoch_idx)
    return best_avg_gap

In [7]:
model = VAE_8(config).to(config.device)
optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)
# training_data = torch.load('training_data.torch')
# training_data[0] = training_data[0].numpy()
# training_data[1] = training_data[1].numpy()

In [8]:
# validation_data = torch.load('validation_data.torch')
# validation_data[0] = validation_data[0].numpy()
# validation_data[1] = validation_data[1].numpy()
training_dataset = TSPDataset(config.epoch_size, config.problem_size, config, training_data)
training_dataloader = DataLoader(training_dataset, batch_size=config.batch_size, num_workers=0, shuffle=True)
validation_dataset = TSPDataset(config.network_validation_size, config.problem_size, config, validation_data)
validation_dataloader = DataLoader(validation_dataset, batch_size=config.batch_size, num_workers=0, shuffle=True)
lr_scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, config.lr, epochs=config.nb_epochs, steps_per_epoch=len(training_dataloader))
# training_instances, training_solutions = training_data
# training_instances = training_instances.float().to(config.device)
# training_solutions = training_solutions.to(config.device)

In [ ]:
avg_gap = do()
writer.add_hparams(config, {"hparam/avg_gap": avg_gap * 100})

  8%|██▉                                 | 49/600 [1:30:47<16:57:07, 110.76s/it]

Costs 3.784010887145996
Costs 4.025845527648926
Costs 4.24570894241333
Costs 3.9037623405456543
Costs 4.680887699127197
Costs 3.835512638092041
Costs 3.8831405639648438
Costs 4.210783004760742
Costs 3.7174248695373535
Costs 3.3524513244628906
Costs 4.231750011444092
Costs 4.261957168579102
Costs 3.982050657272339
Costs 3.9745168685913086
Costs 4.032354831695557
Costs 4.275890350341797
Costs 3.7329540252685547
Costs 4.026113510131836
Costs 4.0115509033203125
Costs 3.405766248703003
Costs 3.167098045349121
Costs 3.4243922233581543
Costs 3.981553792953491
Costs 3.8194570541381836
Costs 3.6772985458374023
Costs 3.7470035552978516
Costs 3.879744291305542
Costs 4.30179500579834
Costs 3.8568358421325684
Costs 3.764086961746216
Costs 4.068649768829346
Costs 3.8368465900421143
Costs 3.7137937545776367
Costs 3.79575514793396
Costs 3.453484535217285
Costs 3.8417587280273438
Costs 4.253318786621094
Costs 3.9465386867523193
Costs 3.3896450996398926
Costs 4.259589195251465
Costs 3.747296094894409
Co

  8%|███                                 | 50/600 [1:53:21<73:52:54, 483.59s/it]

Costs 4.179910659790039


 11%|███▊                                | 64/600 [2:18:47<16:38:28, 111.77s/it]